# LLM Fine-Tuning for Loan Explanations

> **Original source:** [cbtham/micro-financial-loan](https://github.com/cbtham/micro-financial-loan)
> **Author:** cbtham
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit) for dynamic cluster configuration

In [ ]:
# Auto-configure environment for this cluster
# Override any value by creating a .env file in this directory

import os
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)
except ImportError:
    pass

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio.financial-loan-demo.svc:9000")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")
S3_BUCKET_MODELS = os.environ.get("S3_BUCKET_MODELS", "models")
NAMESPACE = os.environ.get("NAMESPACE", "financial-loan-demo")

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")
MODEL_SHORT_NAME = HF_MODEL_ID.split("/")[-1].lower().replace("_", "-")
OUTPUT_DIR = f"./{MODEL_SHORT_NAME}-loan-explanations"
S3_MODEL_PREFIX = f"{MODEL_SHORT_NAME}-loan-explanations/"

print("=" * 60)
print("Environment Configuration")
print("=" * 60)
print(f"  S3 Endpoint:    {S3_ENDPOINT}")
print(f"  Models Bucket:  {S3_BUCKET_MODELS}")
print(f"  Namespace:      {NAMESPACE}")
print(f"  HF Model:       {HF_MODEL_ID}")
print(f"  Output Dir:     {OUTPUT_DIR}")
print("=" * 60)
print("To override, create a .env file or set environment variables")

# LLM Model Fine-tuning for Microloan Explanations

This notebook fine-tunes a causal LM (default: **Qwen3-4B-Instruct-2507**) for generating loan approval/rejection explanations.
Set `HF_MODEL_ID` in your `.env` file to use a different model.

**Why Qwen3-4B (default)?**
- Optimized for L4 GPU (4B params = ~8GB VRAM)
- vLLM compatible (fast inference with OpenAI API)
- Superior reasoning and instruction following
- Easy deployment on KServe with vLLM runtime

**Training Time:** ~20-30 minutes on L4 GPU with LoRA

## 1. Install Dependencies

In [ ]:
# Install dependencies
!pip install -q transformers>=4.51.0 datasets torch accelerate peft bitsandbytes boto3 trl

# Verify versions
import transformers
import torch
print(f"Transformers: {transformers.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

Using device: cuda
GPU: NVIDIA L4
GPU Memory: 23.66 GB


## 2. Import Libraries

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
import json
import os

model_name = HF_MODEL_ID
print(f"Loading model: {model_name}")
print("This may take 2-3 minutes...")

## Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Required for training

## Load model in BF16 for L4 GPU
try:
    # Try with FlashAttention-2 first (fastest)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="flash_attention_2"
    )
    print("Using FlashAttention-2 (optimal)")
except Exception as e:
    print(f"FlashAttention-2 not available, using eager mode: {e}")
    # Fallback to standard attention
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

print(f"Model loaded on: {model.device}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model dtype: {model.dtype}")

Loading model...
This may take 2-3 minutes...


`torch_dtype` is deprecated! Use `dtype` instead!


FlashAttention-2 not available, using eager mode: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on: cuda:0
Model parameters: 4,022,468,096
Model dtype: torch.bfloat16


## 3. Load Training Data from Predictive Model

In [4]:
# Load the feature_explanations_dataset.json generated by the predictive model
import json

# Path to the dataset (generated by PredictiveModelWorkbench/model-training-v3.ipynb)
dataset_path = "./PredictiveModelWorkbench/feature_explanations_dataset.json"

if os.path.exists(dataset_path):
    with open(dataset_path, 'r') as f:
        training_data = json.load(f)
    print(f"Loaded {len(training_data)} training examples from predictive model")
    print("\nSample:")
    print(f"Input:  {training_data[0]['input']}")
    print(f"Output: {training_data[0]['output']}")
else:
    print(f"ERROR: {dataset_path} not found!")
    print("Please run PredictiveModelWorkbench/model-training-v3.ipynb first to generate training data.")

Loaded 2000 training examples from predictive model

Sample:
Input:  Explain loan decision: credit_score=420, income=135000, dti=474.1, loan_amount=640080, decision=rejected
Output: Loan rejected. High default risk (58.0%) identified. Income $135,000 insufficient for loan amount $640,080, resulting in concerning DTI of 474.1%. Established employment history of 20 years demonstrates stability. monthly payments would consume 474% of income


## 4. Format Data for Chat Template

In [5]:
# Format data as chat turns with system/user/assistant roles
formatted_data = []

SYSTEM_PROMPT = """You are an AI loan officer assistant. Explain loan decisions clearly and professionally based on applicant data."""

for item in training_data:
    # Convert to chat format
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["input"]},
        {"role": "assistant", "content": item["output"]}
    ]
    formatted_data.append({"messages": chat})

print(f"Formatted {len(formatted_data)} examples for training")
print("\nSample formatted example:")
print(formatted_data[0])

Formatted 2000 examples for training

Sample formatted example:
{'messages': [{'role': 'system', 'content': 'You are an AI loan officer assistant. Explain loan decisions clearly and professionally based on applicant data.'}, {'role': 'user', 'content': 'Explain loan decision: credit_score=420, income=135000, dti=474.1, loan_amount=640080, decision=rejected'}, {'role': 'assistant', 'content': 'Loan rejected. High default risk (58.0%) identified. Income $135,000 insufficient for loan amount $640,080, resulting in concerning DTI of 474.1%. Established employment history of 20 years demonstrates stability. monthly payments would consume 474% of income'}]}


## 5. Load Model and Tokenizer

In [6]:
model_name = HF_MODEL_ID
print(f"Loading model: {model_name}")
print("This may take 2-3 minutes...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Required for training

# Load model in BF16 for L4 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Model loaded on: {model.device}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model dtype: {model.dtype}")

Loading model...
This may take 2-3 minutes...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on: cuda:0
Model parameters: 4,022,468,096
Model dtype: torch.bfloat16


## 6. Configure LoRA for Efficient Fine-tuning

In [7]:
# LoRA configuration for fine-tuning
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Prepare model for training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Enable training mode
model.train()

# Verify trainable parameters
print("LoRA configuration applied")
model.print_trainable_parameters()

# Verify gradients are enabled
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f" Trainable: {name}")
        break

LoRA configuration applied
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
 Trainable: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight


## 7. Prepare Dataset with Chat Template

In [8]:
# Create HuggingFace Dataset
dataset = Dataset.from_list(formatted_data)

# Tokenize function
def tokenize_function(examples):
    # Apply chat template and tokenize
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    
    # Tokenize
    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors=None
    )
    
    # For causal LM, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# Tokenize dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["messages"]
)

# Split into train/eval
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print(f"Columns: {train_dataset.column_names}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training samples: 1800
Evaluation samples: 200
Columns: ['input_ids', 'attention_mask', 'labels']


## 8. Configure Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./fine-tuned-checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=4,  # L4 can handle this with 4B model
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Effective batch size = 8
    learning_rate=2e-4,
    bf16=True,  # BF16 for L4 GPU (Ada Lovelace)
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",  # Faster optimizer for L4
    report_to="none",  # Disable wandb/tensorboard
    gradient_checkpointing=True,  # Save VRAM
)

print("Training arguments configured for L4 GPU")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

Training arguments configured for L4 GPU
Effective batch size: 8


## 9. Initialize Supervised Fine Tuning SFT Trainer

In [10]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Use standard Trainer (not SFTTrainer for pre-tokenized data)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Trainer initialized")
print(f"Training will take approximately 20-30 minutes on L4 GPU")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer initialized
Training will take approximately 20-30 minutes on L4 GPU


## 10. Fine Tuning LLM Model - Training

In [ ]:
print("Starting training...")
print("="*80)

trainer.train()

print("="*80)
print("Training complete!")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Starting training...


Epoch,Training Loss,Validation Loss


## 11. Merge LoRA Weights and Save Model

In [ ]:
print("Merging LoRA weights into base model...")

# Merge LoRA adapters
merged_model = model.merge_and_unload()

# Save merged model
output_dir = OUTPUT_DIR
merged_model.save_pretrained(output_dir, safe_serialization=True)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")
print(f"Files: {os.listdir(output_dir)}")

## 12. Test the Fine-tuned Model

In [ ]:
print("Testing fine-tuned model...")
print("="*80)

test_cases = [
    "Explain loan decision: credit_score=780, income=85000, dti=22.0, loan_amount=25000, decision=approved",
    "Explain loan decision: credit_score=540, income=32000, dti=58.0, loan_amount=18000, decision=rejected",
]

for test_input in test_cases:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_input}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    
    print(f"\nInput: {test_input}")
    print(f"Response: {response}")
    print("-"*80)

print("="*80)
print("Testing complete!")

## 13. Upload Model to MinIO

In [ ]:
import boto3
from botocore.client import Config
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# MinIO Configuration
s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
    verify=False
)

print("="*80)
print("UPLOADING FINE-TUNED MODEL TO MINIO")
print("="*80)
print("Estimated time: 2-3 minutes for 4B model")

# Configuration
local_model_dir = OUTPUT_DIR
bucket_name = "models"
s3_prefix = S3_MODEL_PREFIX

# Get all files to upload
all_files = []
for root, dirs, files in os.walk(local_model_dir):
    for file in files:
        local_file = os.path.join(root, file)
        relative_path = os.path.relpath(local_file, local_model_dir)
        s3_key = s3_prefix + relative_path
        all_files.append((local_file, s3_key))

total_files = len(all_files)
total_size = sum(os.path.getsize(f[0]) for f in all_files)

print(f"\nFound {total_files} files to upload")
print(f"Total size: {total_size/1e6:.2f} MB")
print(f"Destination: s3://{bucket_name}/{s3_prefix}")
print()

# Upload each file
for idx, (local_file, s3_key) in enumerate(all_files, 1):
    file_size = os.path.getsize(local_file)
    relative_path = os.path.relpath(local_file, local_model_dir)
    progress_pct = (idx / total_files) * 100
    print(f"[{idx}/{total_files}] ({progress_pct:.1f}%) Uploading: {relative_path} ({file_size/1e6:.1f} MB)")
    
    s3.upload_file(local_file, bucket_name, s3_key)

print()
print("="*80)
print("UPLOAD COMPLETE!")
print("="*80)
print(f"Total files uploaded: {total_files}")
print(f"Total size: {total_size/1e6:.2f} MB")
print(f"Model location: s3://{bucket_name}/{s3_prefix}")
print()
print("Next steps:")
print("1. Deploy using vLLM runtime (much faster than T5!)")
print(f"2. Use path: {S3_MODEL_PREFIX.rstrip('/')}")
print("3. API endpoint: /v1/chat/completions (OpenAI-compatible)")
print("="*80)

#### Next Deploy with vLLM Runtime in Red Hat AI